# Phase 5 — Data Preprocessing

## Objective

Prepare the engineered dataset for machine learning by:
- Removing unnecessary/helper features
- Handling missing values
- Encoding categorical variables
- Creating a reproducible preprocessing pipeline

In [14]:
import numpy as np
import pandas as pd

In [25]:
train_engineered = pd.read_csv(
    "../data/processed/train_engineered.csv",parse_dates=["Date"]
)

C:\Users\ragha\AppData\Local\Temp\ipykernel_17664\3021023370.py:1: DtypeWarning: Columns (0: StateHoliday) have mixed types. Specify dtype option on import or set low_memory=False.
  train_engineered = pd.read_csv(


In [26]:
train_engineered.shape

(844392, 34)

In [27]:
train_engineered.info()

<class 'pandas.DataFrame'>
RangeIndex: 844392 entries, 0 to 844391
Data columns (total 34 columns):
 #   Column                     Non-Null Count   Dtype         
---  ------                     --------------   -----         
 0   Store                      844392 non-null  int64         
 1   DayOfWeek                  844392 non-null  int64         
 2   Date                       844392 non-null  datetime64[us]
 3   Sales                      844392 non-null  int64         
 4   Customers                  844392 non-null  int64         
 5   Open                       844392 non-null  int64         
 6   Promo                      844392 non-null  int64         
 7   StateHoliday               844392 non-null  object        
 8   SchoolHoliday              844392 non-null  int64         
 9   StoreType                  844392 non-null  str           
 10  Assortment                 844392 non-null  str           
 11  CompetitionDistance        842206 non-null  float64       
 12 

In [28]:
train_engineered.isna().sum().sort_values(ascending=False)

Promo2SinceWeek              423307
PromoInterval                423307
Promo2StartDate              423307
PromoAgeMonths               423307
Promo2SinceYear              423307
CompetitionOpenDate          268619
CompetitionOpenSinceMonth    268619
CompetitionOpenSinceYear     268619
CompetitionDistance            2186
Store                             0
DayOfWeek                         0
StoreType                         0
StateHoliday                      0
Promo                             0
Open                              0
SchoolHoliday                     0
Date                              0
Sales                             0
Promo2                            0
Assortment                        0
Customers                         0
Year                              0
Quarter                           0
Month                             0
Day                               0
WeekOfYear                        0
IsWeekend                         0
IsMonthEnd                  

## 1. Removing Unnecessary Features

Some columns created during feature engineering were helper variables used only for deriving meaningful features.

These columns do not provide additional predictive value and are removed before modeling.

In [29]:
columns_to_remove = [
    "Customers",
    "CompetitionOpenSinceMonth",
    "CompetitionOpenSinceYear",
    "CompetitionOpenDate",
    "Promo2SinceWeek",
    "Promo2SinceYear",
    "Promo2StartDate",
    "PromoInterval",
    "PromoAgeMonths",
    "TransactionMonth",
    "Open"
]

missing_columns = set(columns_to_remove) - set(train_engineered.columns)

print("Missing columns:", missing_columns)

Missing columns: set()


### Keeping Date to avoid temporal leakage.

In [30]:
train_engineered.drop(columns=columns_to_remove, inplace=True)

In [31]:
train_engineered.shape

(844392, 23)

In [32]:
train_engineered.info()

<class 'pandas.DataFrame'>
RangeIndex: 844392 entries, 0 to 844391
Data columns (total 23 columns):
 #   Column                    Non-Null Count   Dtype         
---  ------                    --------------   -----         
 0   Store                     844392 non-null  int64         
 1   DayOfWeek                 844392 non-null  int64         
 2   Date                      844392 non-null  datetime64[us]
 3   Sales                     844392 non-null  int64         
 4   Promo                     844392 non-null  int64         
 5   StateHoliday              844392 non-null  object        
 6   SchoolHoliday             844392 non-null  int64         
 7   StoreType                 844392 non-null  str           
 8   Assortment                844392 non-null  str           
 9   CompetitionDistance       842206 non-null  float64       
 10  Promo2                    844392 non-null  int64         
 11  Year                      844392 non-null  int64         
 12  Month        

In [34]:
train_engineered["Date"].describe()

count                        844392
mean     2014-04-11 01:02:42.487564
min             2013-01-01 00:00:00
25%             2013-08-16 00:00:00
50%             2014-03-31 00:00:00
75%             2014-12-10 00:00:00
max             2015-07-31 00:00:00
Name: Date, dtype: object

In [35]:
train_engineered = train_engineered.sort_values("Date").reset_index(drop=True)

In [40]:
cutoff_date="2015-06-01"
train_df=train_engineered[train_engineered["Date"]<cutoff_date]
valid_df=train_engineered[train_engineered["Date"]>=cutoff_date]

In [41]:
print(train_df["Date"].describe())
print(valid_df["Date"].describe())

count                        785781
mean     2014-03-08 18:18:21.989485
min             2013-01-01 00:00:00
25%             2013-08-01 00:00:00
50%             2014-02-28 00:00:00
75%             2014-10-16 00:00:00
max             2015-05-31 00:00:00
Name: Date, dtype: object
count                         58611
mean     2015-07-01 05:30:43.210319
min             2015-06-01 00:00:00
25%             2015-06-16 00:00:00
50%             2015-07-01 00:00:00
75%             2015-07-16 00:00:00
max             2015-07-31 00:00:00
Name: Date, dtype: object


In [44]:
set(train_df["Date"])&set(valid_df["Date"])

set()

In [45]:
train_df.isna().sum().sort_values(ascending=False)

CompetitionDistance         2028
DayOfWeek                      0
Store                          0
Sales                          0
Promo                          0
StateHoliday                   0
Date                           0
SchoolHoliday                  0
StoreType                      0
Assortment                     0
Promo2                         0
Year                           0
Month                          0
Day                            0
WeekOfYear                     0
Quarter                        0
IsMonthStart                   0
IsMonthEnd                     0
IsWeekend                      0
CompetitionInfoAvailable       0
CompetitionAgeMonths           0
Promo2AgeMonths                0
IsPromoMonth                   0
dtype: int64

## Fixing Mising "CompetitionDistance"
 Missing values in CompetitionDistance represented unavailable competition distance information rather than measurement errors. Since only ~0.26% of observations were missing and -1 is outside the valid domain of distances, sentinel value imputation was used to preserve the missingness information without introducing an additional indicator feature

In [48]:
train_df.loc[
    train_df["CompetitionDistance"].isna(),
    "CompetitionDistance"
] = -1

valid_df.loc[
    valid_df["CompetitionDistance"].isna(),
    "CompetitionDistance"
] = -1

In [49]:
print(train_df["CompetitionDistance"].isna().sum())
print(valid_df["CompetitionDistance"].isna().sum())

0
0


In [56]:
X_train=train_df.drop(columns="Sales")
y_train=train_df["Sales"]

X_valid=valid_df.drop(columns="Sales")
y_valid=valid_df["Sales"]

In [57]:
X_train.info()

<class 'pandas.DataFrame'>
RangeIndex: 785781 entries, 0 to 785780
Data columns (total 22 columns):
 #   Column                    Non-Null Count   Dtype         
---  ------                    --------------   -----         
 0   Store                     785781 non-null  int64         
 1   DayOfWeek                 785781 non-null  int64         
 2   Date                      785781 non-null  datetime64[us]
 3   Promo                     785781 non-null  int64         
 4   StateHoliday              785781 non-null  object        
 5   SchoolHoliday             785781 non-null  int64         
 6   StoreType                 785781 non-null  str           
 7   Assortment                785781 non-null  str           
 8   CompetitionDistance       785781 non-null  float64       
 9   Promo2                    785781 non-null  int64         
 10  Year                      785781 non-null  int64         
 11  Month                     785781 non-null  int64         
 12  Day          

In [58]:
numerical_features = ["CompetitionDistance","Year","CompetitionAgeMonths","Promo2AgeMonths","Day","WeekOfYear"]
categorical_features = ["DayOfWeek","StateHoliday","StoreType","Assortment","Month","Quarter"]
binary_features = ["Promo","SchoolHoliday","Promo2","IsMonthStart","IsMonthEnd","IsWeekend","CompetitionInfoAvailable","IsPromoMonth"]
identifier_features = ["Store"]

In [60]:
X_train=X_train.drop(columns="Date")
X_valid=X_valid.drop(columns="Date")

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

count     785781
unique         5
top            0
freq      475136
Name: StateHoliday, dtype: int64

In [71]:
X_train["StateHoliday"]=X_train["StateHoliday"].astype(str)
X_valid["StateHoliday"]=X_valid["StateHoliday"].astype(str)
one_hot_features = [
    "StateHoliday",
    "StoreType",
    "Assortment"
]

In [72]:
preprocessor=ColumnTransformer(
    transformers=[(
        "categories",
        OneHotEncoder(handle_unknown="ignore",sparse_output=False),
        one_hot_features
)],
    remainder="passthrough"
    )

In [73]:
preprocessor.fit(X_train)

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('categories', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'passthrough'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``feature_name``

In [74]:
preprocessor.named_transformers_["categories"]

,"sparse_output sparse_output: bool, default=TrueWhen ``True``, it returns a SciPy sparse matrix/arrayin ""Compressed Sparse Row"" (CSR) format... versionadded:: 1.2 `sparse` was renamed to `sparse_output`",False
,"handle_unknown handle_unknown: {'error', 'ignore', 'infrequent_if_exist', 'warn'}, default='error'Specifies the way unknown categories are handled during :meth:`transform`.- 'error' : Raise an error if an unknown category is present during transform.- 'ignore' : When an unknown category is encountered during transform, the resulting one-hot encoded columns for this feature will be all zeros. In the inverse transform, an unknown category will be denoted as None.- 'infrequent_if_exist' : When an unknown category is encountered during transform, the resulting one-hot encoded columns for this feature will map to the infrequent category if it exists. The infrequent category will be mapped to the last position in the encoding. During inverse transform, an unknown category will be mapped to the category denoted `'infrequent'` if it exists. If the `'infrequent'` category does not exist, then :meth:`transform` and :meth:`inverse_transform` will handle an unknown category as with `handle_unknown='ignore'`. Infrequent categories exist based on `min_frequency` and `max_categories`. Read more in the :ref:`User Guide <encoder_infrequent_categories>`.- 'warn' : When an unknown category is encountered during transform a warning is issued, and the encoding then proceeds as described for `handle_unknown=""infrequent_if_exist""`... versionchanged:: 1.1 `'infrequent_if_exist'` was added to automatically handle unknown categories and infrequent categories... versionadded:: 1.6 The option `""warn""` was added in 1.6.",'ignore'
,"categories categories: 'auto' or a list of array-like, default='auto'Categories (unique values) per feature:- 'auto' : Determine categories automatically from the training data.- list : ``categories[i]`` holds the categories expected in the ith column. The passed categories should not mix strings and numeric values within a single feature, and should be sorted in case of numeric values.The used categories can be found in the ``categories_`` attribute... versionadded:: 0.20",'auto'
,"drop drop: {'first', 'if_binary'} or an array-like of shape (n_features,), default=NoneSpecifies a methodology to use to drop one of the categories perfeature. This is useful in situations where perfectly collinearfeatures cause problems, such as when feeding the resulting datainto an unregularized linear regression model.However, dropping one category breaks the symmetry of the originalrepresentation and can therefore induce a bias in downstream models,for instance for penalized linear classification or regression models.- None : retain all features (the default).- 'first' : drop the first category in each feature. If only one category is present, the feature will be dropped entirely.- 'if_binary' : drop the first category in each feature with two categories. Features with 1 or more than 2 categories are left intact.- array : ``drop[i]`` is the category in feature ``X[:, i]`` that should be dropped.When `max_categories` or `min_frequency` is configured to groupinfrequent categories, the dropping behavior is handled after thegrouping... versionadded:: 0.21 The parameter `drop` was added in 0.21... versionchanged:: 0.23 The option `drop='if_binary'` was added in 0.23... versionchanged:: 1.1 Support for dropping infrequent categories.",None
,"dtype dtype: number type, default=np.float64Desired dtype of output.",<class 'numpy.float64'>
,"min_frequency min_frequency: int or float, default=NoneSpecifies the minimum frequency below which a category will beconsidered infrequent.- If `int`, categories with a smaller cardinality will be considered infrequent.- If `float`, categories with a smaller cardinality than `min_frequency * n_samples` will be considered infrequent... versionadded:: 1.1 Read more in the :ref:`User Guide <encoder_infreque

In [75]:
preprocessor.named_transformers_["categories"].categories_

[array(['0', 'a', 'b', 'c'], dtype=object),
 array(['a', 'b', 'c', 'd'], dtype=object),
 array(['a', 'b', 'c'], dtype=object)]

In [76]:
X_train_processed = preprocessor.transform(X_train)
X_valid_processed = preprocessor.transform(X_valid)

In [77]:
preprocessor.get_feature_names_out()

array(['categories__StateHoliday_0', 'categories__StateHoliday_a',
       'categories__StateHoliday_b', 'categories__StateHoliday_c',
       'categories__StoreType_a', 'categories__StoreType_b',
       'categories__StoreType_c', 'categories__StoreType_d',
       'categories__Assortment_a', 'categories__Assortment_b',
       'categories__Assortment_c', 'remainder__Store',
       'remainder__DayOfWeek', 'remainder__Promo',
       'remainder__SchoolHoliday', 'remainder__CompetitionDistance',
       'remainder__Promo2', 'remainder__Year', 'remainder__Month',
       'remainder__Day', 'remainder__WeekOfYear', 'remainder__Quarter',
       'remainder__IsMonthStart', 'remainder__IsMonthEnd',
       'remainder__IsWeekend', 'remainder__CompetitionInfoAvailable',
       'remainder__CompetitionAgeMonths', 'remainder__Promo2AgeMonths',
       'remainder__IsPromoMonth'], dtype=object)

In [78]:
X_train_processed_df = pd.DataFrame(
    X_train_processed,
    columns=preprocessor.get_feature_names_out(),
    index=X_train.index
)

In [80]:
display(X_train_processed_df.info())

<class 'pandas.DataFrame'>
RangeIndex: 785781 entries, 0 to 785780
Data columns (total 29 columns):
 #   Column                               Non-Null Count   Dtype  
---  ------                               --------------   -----  
 0   categories__StateHoliday_0           785781 non-null  float64
 1   categories__StateHoliday_a           785781 non-null  float64
 2   categories__StateHoliday_b           785781 non-null  float64
 3   categories__StateHoliday_c           785781 non-null  float64
 4   categories__StoreType_a              785781 non-null  float64
 5   categories__StoreType_b              785781 non-null  float64
 6   categories__StoreType_c              785781 non-null  float64
 7   categories__StoreType_d              785781 non-null  float64
 8   categories__Assortment_a             785781 non-null  float64
 9   categories__Assortment_b             785781 non-null  float64
 10  categories__Assortment_c             785781 non-null  float64
 11  remainder__Store        

None

In [84]:
print(X_train_processed.shape)
print(X_valid_processed.shape)

X_train_processed_df.head()


(785781, 29)
(58611, 29)


,categories__StateHoliday_0,categories__StateHoliday_a,categories__StateHoliday_b,categories__StateHoliday_c,categories__StoreType_a,categories__StoreType_b,categories__StoreType_c,categories__StoreType_d,categories__Assortment_a,categories__Assortment_b,...,remainder__Day,remainder__WeekOfYear,remainder__Quarter,remainder__IsMonthStart,remainder__IsMonthEnd,remainder__IsWeekend,remainder__CompetitionInfoAvailable,remainder__CompetitionAgeMonths,remainder__Promo2AgeMonths,remainder__IsPromoMonth
0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,...,1.0,1.0,1.0,1.0,0.0,0.0,1.0,130.0,-1.0,0.0
1,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,...,1.0,1.0,1.0,1.0,0.0,0.0,1.0,15.0,-1.0,0.0
2,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,...,1.0,1.0,1.0,1.0,0.0,0.0,0.0,-1.0,-1.0,0.0
3,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,...,1.0,1.0,1.0,1.0,0.0,0.0,1.0,0.0,-1.0,0.0
4,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,...,1.0,1.0,1.0,1.0,0.0,0.0,0.0,-1.0,0.0,1.0


In [85]:
import joblib

joblib.dump(preprocessor, "../models/preprocessor.joblib")

['../models/preprocessor.joblib']